In [14]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Input, Model

In [15]:
X, y = make_classification(
    n_samples= 1000, 
    n_features= 20,
    n_informative= 10,
    n_classes= 2,
    random_state= 42
)

In [16]:
X.shape, y.shape

((1000, 20), (1000,))

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)
X_train.shape, X_test.shape

((800, 20), (200, 20))

In [18]:
# ── Sequential API ──────────────────────────────────────────
# Stack layers in a list — Keras wires them automatically.

model_seq = keras.Sequential([
    layers.Dense(64, activation= 'relu', input_shape = (X_train.shape[1], )),
    layers.Dropout(0.2),
    layers.Dense(32, activation= 'relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation= 'sigmoid')
])

early_stop = keras.callbacks.EarlyStopping(
    monitor= 'val_loss',
    patience = 5,
    restore_best_weights=True
)

In [19]:
model_seq.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

In [20]:
model_seq.fit(X_train, y_train, epochs = 20, batch_size= 32, verbose = 1, callbacks = [early_stop])
loss, acc = model_seq.evaluate(X_test, y_test, verbose= 0)
print(f'Accuracy : {acc:.2f}')

Epoch 1/20


25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 788us/step - accuracy: 0.6300 - loss: 0.6735 
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 917us/step - accuracy: 0.8000 - loss: 0.4668
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 786us/step - accuracy: 0.8263 - loss: 0.3855
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 700us/step - accuracy: 0.8675 - loss: 0.3070
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 627us/step - accuracy: 0.9013 - loss: 0.2873
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 629us/step - accuracy: 0.9175 - loss: 0.2358
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 628us/step - accuracy: 0.9162 - loss: 0.2238
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 629us/step - accuracy: 0.9137 - loss: 0.2144
Epoch 9/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 627us/step - accuracy: 0.9287 - loss: 0.2006
Epoch 10/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 682us/step - accuracy: 0.9187 - loss: 0.1902
Epoch 11/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 687us/step - accuracy: 0.9275 - loss: 0.1764
Epoch 12/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 682us/step - accurac

In [21]:
# ── Functional API ──────────────────────────────────────────
# Wire tensors explicitly — supports multi-input/output models.

inputs = Input(shape = (X_train.shape[1], ))
x = layers.Dense(64, activation= 'relu')(inputs)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation = 'relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation = 'sigmoid')(x)

model_func = Model(inputs = inputs, outputs = outputs)

early_stop = keras.callbacks.EarlyStopping(
    monitor= 'val_loss',
    patience = 5,
    restore_best_weights=True
)

model_func.compile(
    optimizer= 'adam', 
    loss = 'binary_crossentropy',
    metrics= ['accuracy']
)

In [22]:
model_func.fit(X_train, y_train, epochs = 20, batch_size= 32, verbose = 0, callbacks= [early_stop])

loss, acc = model_func.evaluate(X_test, y_test, verbose = 0)
print(f'Accuracy : {acc:.4f}')

Accuracy : 0.9500


In [23]:
# Model Subclass API 

In [ ]:
# ── Model Subclass API ─────────────────────────────────────────
# Maximum flexibility — define any forward logic you like.
# Layers go in __init__, forward pass logic goes in call().

class MyModel(tf.keras.Model):

    def __init__(self):
        super().__init__()
        self.d1 = layers.Dense(64, activation='relu')
        self.drop1 = layers.Dropout(0.2)
        self.d2 = layers.Dense(32, activation='relu')
        self.drop2 = layers.Dropout(0.2)
        self.out = layers.Dense(1, activation='sigmoid')

    def call(self, inputs, training=False):
        x = self.d1(inputs)
        x = self.drop1(x, training=training)
        x = self.d2(x)
        x = self.drop2(x, training=training)
        return self.out(x)

model = MyModel()

In [25]:
model.compile(
    optimizer= 'adam', 
    loss = 'binary_crossentropy',
    metrics= ['accuracy']
)

In [ ]:
model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=0,
    callbacks=[early_stop]
)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Subclass API — Accuracy: {acc:.4f}')